In [1]:
import pandas as pd
import numpy as np
import math

# google_trends = pd.read_csv("data/gold_google_trends_daily.csv")
data = pd.read_csv("../files/processed_data.csv")
match_context = pd.read_csv("../data/gold_match_context.csv")
# match_goals = pd.read_csv("data/gold_match_goals.csv")
match_tickets = pd.read_csv("../data/gold_match_tickets.csv")
# matches = pd.read_csv("data/gold_match.csv")
matches = pd.read_csv("../data/gold_match_with_league_standings.csv")
match_articles = pd.read_csv("../data/gold_belga_press_articles.csv", on_bad_lines="skip")

In [2]:
article_count = (match_articles
                 .groupby("match_id")["match_id"]
                 .value_counts()
                 .to_frame()
                 .reset_index()
                 .rename(columns={"count": "article_count"}))

In [3]:
matches["league_pos_home_inv"] = 19 - matches["league_position_home"]
matches["league_pos_away_inv"] = 19 - matches["league_position_away"]

In [4]:
tickets_sold_goals = match_tickets[["tickets_sold_total", "match_id", "seasonpass_holders"]]
matchdays = matches[["matchday", "match_id"]]
league_positions = matches[["league_pos_home_inv", "league_pos_away_inv", "match_id"]]


data = pd.merge(data, tickets_sold_goals, on="match_id")
data = pd.merge(data, matchdays, on="match_id")
data = pd.merge(data, league_positions, on="match_id")
data["away_team_code"] = data["away_team_code"].astype("category")


In [5]:
data["date"] = pd.to_datetime(data["date"])
data["month"] = data['date'].dt.month
data['month_cos'] = np.cos(2 * np.pi * data['month'] / 12)
# data = data.drop(["date", "month"], axis=1)

In [6]:
data["kickoff_time_local"] = pd.to_datetime(data["kickoff_time_local"], format='%H:%M:%S')
data["kickoff_time_local"] = pd.to_datetime(data["kickoff_time_local"], format='%H:%M:%S').dt.hour
data["is_18_hours"] = data["kickoff_time_local"] == 18
# data = data.drop("kickoff_time_local", axis=1)

In [7]:
data["is_sunday"] = data["weekday"] == 6
data["is_fri_hol"] = data["weekday"].isin([4, 5, 6])

In [8]:
n = 15
train_data = data[:-n]
test_data = data.tail(n)

In [9]:
avg_tickets_scanned = train_data.groupby("away_team_code")["tickets_scanned"].mean().to_frame().rename(columns={"tickets_scanned": "avg_tickets_scanned"})
match_context_part = match_context[["academic_week", "has_promotion", "match_id", "is_public_holiday"]]

In [10]:
train_data = pd.merge(train_data, match_context_part, on="match_id")
train_data = pd.merge(train_data, article_count, on="match_id")
train_data = pd.merge(train_data, avg_tickets_scanned, on="away_team_code")

test_data = pd.merge(test_data, match_context_part, on="match_id")
test_data = pd.merge(test_data, article_count, on="match_id")
test_data = pd.merge(test_data, avg_tickets_scanned, on="away_team_code")

In [11]:
train_data = train_data.drop("match_id", axis=1)
test_data = test_data.drop("match_id", axis=1)

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error

X_train = train_data.drop(["tickets_scanned", "kickoff_time_local", "date"], axis=1)
y_train = train_data["tickets_scanned"]

X_test = test_data.drop(["tickets_scanned", "kickoff_time_local", "date"], axis=1)
y_test = test_data["tickets_scanned"]

num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
bool_cols = X_train.select_dtypes(include=["bool"]).columns
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(), cat_cols),
        ("bool", "passthrough", bool_cols)
    ]
)

model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("regressor", LinearRegression())
])

model.fit(X_train, y_train)

preds = model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
print("MAE", mae)


MAE 1480.999084122548


In [15]:
# Get feature names after preprocessing
feature_names = model.named_steps["preprocessing"].get_feature_names_out()

# Get coefficients
coefficients = model.named_steps["regressor"].coef_

# Combine into DataFrame
coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
}).sort_values(by="coefficient", key=abs, ascending=False)

# print(coef_df.head(15))


TypeError: DataFrame.sort_values() missing 1 required positional argument: 'by'

In [17]:
coef_df.sort_values(ascending=False, by='coefficient')

,feature,coefficient
36,bool__is_public_holiday,3770.688286
25,cat__away_team_code_RWD,2622.641637
22,cat__away_team_code_KVK,1442.456854
3,num__tickets_sold_total,1233.705472
11,num__avg_tickets_scanned,1001.451418
15,cat__away_team_code_CER,900.709798
32,bool__is_18_hours,824.612944
34,bool__is_fri_hol,797.823214
5,num__matchday,505.629230
0,num__last_result_vs_opponent,414.857741
